# Week 7 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-07.md`](../../weeks/week-07.md) · **Given engine:** [`waf.py`](waf.py)

Teaching walkthrough of the week-7 studio (blocklists lose, boundaries win). It
**imports the reference answers from [`solution.py`](solution.py)** — never re-pasting
them — so what runs here is exactly what `test_waf.py` grades. Verify:
`python3 studios/_verify_solutions.py week-07`.

Laptop-only: pure-Python measurements plus the week-6 `vuln-web` app run **in-process**
(stdlib `http.server`, a free 127.0.0.1 port, no Docker).

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: week folder (for solution + engine) + repo root (for seclab) ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-07"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import solution
from waf import (KNOWN_ATTACKS, BYPASSES, load_benign, start_vulnweb,
                 vulnweb_canary, waf_block, anomaly)
BENIGN = load_benign()
print(f"loaded {len(KNOWN_ATTACKS)} known attacks, {len(BYPASSES)} bypasses, "
      f"{len(BENIGN)} benign inputs (reused Duel-2 corpus)")

## Task 1 — coverage on KNOWN attacks: the trap

`measure_coverage` on `KNOWN_ATTACKS` is **4/4**. That *looks* like security — but it
only measures the payloads you already thought of. Coverage on known attacks tells you
nothing about what you didn't enumerate. Mirrors
`test_coverage_on_known_attacks_looks_perfect`.

In [ ]:
print(inspect.getsource(solution.measure_coverage))
blocked, total = solution.measure_coverage(KNOWN_ATTACKS)
assert (blocked, total) == (4, 4)
print(f"WAF coverage on KNOWN attacks: {blocked}/{total} -- 'looks great' (the trap)")

## Task 2 — bypass it (axis 4): same intent, new surface

Each bypass is a five-minute mutation: `OR 'a'='a'` instead of `OR 1=1`, a case/spacing
`UnIoN SeLeCt`, an `<img onerror>` with no `<script`, a zero-width char splitting `DROP`.
**3/4 get through.** A defense you haven't bypassed is unevaluated. Mirrors
`test_blocklist_is_bypassed_axis_4`.

In [ ]:
print(inspect.getsource(solution.measure_bypasses))
through, total = solution.measure_bypasses(BYPASSES)
assert through == 3
for p, why in BYPASSES:
    print(f"  {'BLOCKED' if waf_block(p) else 'PASSED ':8} {p[:34]:36} ({why})")
print(f"\n{through}/{total} bypasses evaded the WAF (axis 4)")

## Task 3 — false positives (axis 5): real users turned away

A rule broad enough to catch variants also blocks **legitimate** traffic. The benign
phrase `drop table tennis lessons for beginners` trips the DROP TABLE rule. Every false
positive is a support ticket and an analyst-hour — and a WAF that blocks users gets
disabled, at which point its coverage is zero. Mirrors
`test_blocklist_false_positives_axis_5`.

In [ ]:
print(inspect.getsource(solution.measure_false_positives))
fps = solution.measure_false_positives(BENIGN)
blocked_inputs = {inp for inp, _ in fps}
assert len(fps) >= 1
assert "drop table tennis lessons for beginners" in blocked_inputs
print(f"WAF false-positived on {len(fps)}/{len(BENIGN)} legit inputs (axis 5):")
for inp, rule in fps:
    print(f"   [{rule}] {inp!r}")

## The guarantee-fail — the blocklist loses on BOTH blades at once

The negative-security promise ("we filter attacks") collapses: the WAF is **bypassable**
(axis 4) AND **noisy** (axis 5), *simultaneously*. Loosen the rules to cut false positives
and the bypasses multiply; tighten them and more real users suffer. There is no complete-
and-quiet setting, because the space of malicious input is open-ended and overlaps benign
text. Stages `test_blocklist_loses_on_both_blades`.

In [ ]:
through, _ = solution.measure_bypasses(BYPASSES)
fps = solution.measure_false_positives(BENIGN)
assert through > 0 and len(fps) > 0
print(f"GUARANTEE-FAIL: {through} bypasses THROUGH (axis 4) AND {len(fps)} false positives "
      f"(axis 5), at once.")
print("LESSON: negative security (enumerate bad) can never be both complete and quiet.")

## Task 4 — the fix: a positive-model guarantee (axis 2)

`parameterized_login` treats input as **data**, never code: usernames index a dict; no
input reaches a query context. So **0/8** injections succeed — not "blocked by a rule"
but *structurally* unable to inject, for ANY input. That is the guarantee the WAF could
never state. Mirrors `test_parameterization_guarantee_holds`.

In [ ]:
print(inspect.getsource(solution.count_injection_successes))
all_payloads = list(KNOWN_ATTACKS) + [p for p, _ in BYPASSES]
succeeded, total = solution.count_injection_successes(all_payloads)
assert total == 8 and succeeded == 0
print(f"parameterization defeated {total}/{total} injections structurally ({succeeded} succeeded)")
print("GUARANTEE (axis 2): user input cannot alter query structure, for any input.")

## Task 5 — the attack moment, on the REAL app, oracle-confirmed

Put the WAF in front of the real `vuln-web` `/login`. A WAF-**passed** bypass reaches the
vulnerable query and leaks the admin canary; the sound oracle checks for the canary token
`vulnweb_canary()` in the response (not the echoed payload). The WAF is a **sensor, not a
wall**. Mirrors `test_bypass_leaks_canary_on_real_app`.

In [ ]:
print(inspect.getsource(solution.bypass_leaks_canary))
base, shutdown = start_vulnweb()
try:
    leaked = solution.bypass_leaks_canary(base)
    assert leaked is True
    flagged = [p for p, _ in BYPASSES if not waf_block(p) and anomaly(p)]
    print(f"ATTACK MOMENT: a WAF-passed bypass leaked the real canary {vulnweb_canary()!r} via /login")
    print(f"              (confirmed by a sound oracle, not a reflected string).")
    print(f"detection: {len(flagged)} of the passed bypasses are still ANOMALOUS -- a sensor, not a wall.")
finally:
    shutdown()

## Control Scorecard (Task 6)

| Control | Model | Guarantee (axis 2) | Bypass (axis 4) | FP cost (axis 5) |
|---|---|---|---|---|
| WAF blocklist | negative (enumerate bad) | **none** | yes — trivial variants (3/4) | high — blocks real users |
| Parameterized queries | positive (input is data) | **yes — input can't become code** | none | zero |
| WAF as detection | — | none, but **visibility** | n/a | analyst load (week 13's paradox) |

**Blocklists lose on both blades; a boundary gives a guarantee.** A WAF is a stopgap and
a sensor (virtual patching, detection) — never a substitute for fixing the code.